# PSTHEstimation

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/PSTHEstimation.md`


Notebook source link: [PSTHEstimation.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/PSTHEstimation.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PSTHEstimation"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PSTHEstimation: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PSTHEstimation: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PSTHEstimation: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PSTHEstimation: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "delta = 0.001;",
    "Tmax = 10;",
    "time = 0:delta:Tmax;",
    "f=.2;",
    "lambdaData = 10*sin(2*pi*f*time)+10; %lambda >=0",
    "lambda = Covariate(time,lambdaData, '\\Lambda(t)','time','s','Hz',{'\\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});",
    "numRealizations = 20; % Use 20 realization so that lamba and raster plot are the same size",
    "spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);",
    "spikeColl.plot; set(gca,'ytickLabel',[]);",
    "lambda.plot;",
    "figure;",
    "binsize = .5; %500ms window",
    "psth    = spikeColl.psth(binsize);",
    "psthGLM = spikeColl.psthGLM(binsize);",
    "trueRate = lambda; %rate*delta = expected number of arrivals per bin",
    "h1=trueRate.plot;",
    "h3=psthGLM.plot([],{{' ''k'',''Linewidth'',4'}});",
    "h2=psth.plot([],{{' ''rx'',''Linewidth'',4'}});",
    "legend off;",
    "legend([h1(1) h2(1)  h3(1)],'true','PSTH','PSTH_{glm}');",
    "psth_mean_hz = mean(psth.data);",
    "psth_glm_mean_hz = mean(psthGLM.data);",
    "lambda_mean_hz = mean(lambda.data);",
    "parity = struct();",
    "parity.psth_mean_hz = psth_mean_hz;",
    "parity.psth_glm_mean_hz = psth_glm_mean_hz;",
    "parity.lambda_mean_hz = lambda_mean_hz;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for PSTHEstimation.")


In [ ]:
# Data-style workflow: trial-to-trial variability and PSTH-like estimates.
dt = 0.001
time = np.arange(0.0, 1.2, dt)
n_trials = 30

rate = 5.0 + 8.0 * (time > 0.35) + 4.0 * np.sin(2.0 * np.pi * 2.0 * time)
rate = np.clip(rate, 0.2, None)

trial_matrix = np.zeros((n_trials, time.size), dtype=float)
for k in range(n_trials):
    jitter = 0.6 + 0.8 * rng.random()
    p = np.clip(rate * jitter * dt, 0.0, 0.6)
    trial_matrix[k, :] = rng.binomial(1, p)

psth = trial_matrix.mean(axis=0) / dt
sem = trial_matrix.std(axis=0, ddof=1) / np.sqrt(n_trials) / dt

rates, prob_mat, sig_mat = DecodingAlgorithms.compute_spike_rate_cis(trial_matrix)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
for k in range(min(18, n_trials)):
    t_spk = time[trial_matrix[k] > 0]
    axes[0].vlines(t_spk, k + 0.6, k + 1.4, linewidth=0.5)
axes[0].set_title(f"{TOPIC}: trial raster")
axes[0].set_ylabel("trial")

axes[1].plot(time, psth, color="tab:blue", linewidth=1.2)
axes[1].fill_between(time, psth - sem, psth + sem, color="tab:blue", alpha=0.2)
axes[1].set_ylabel("Hz")
axes[1].set_title("PSTH mean +/- SEM")

im = axes[2].imshow(prob_mat, aspect="auto", origin="lower", cmap="viridis")
axes[2].set_title("Trial-by-trial spike-rate p-values")
axes[2].set_xlabel("trial")
axes[2].set_ylabel("trial")
fig.colorbar(im, ax=axes[2], fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

print("significant pair count", int(sig_mat.sum()))
assert np.allclose(prob_mat, prob_mat.T, atol=1e-12)
assert np.all(np.diag(prob_mat) == 1.0)

CHECKPOINT_METRICS = {
    "psth_mean_hz": float(np.mean(psth)),
    "significant_pairs": float(np.sum(sig_mat)),
}
CHECKPOINT_LIMITS = {
    "psth_mean_hz": (0.1, 50.0),
    "significant_pairs": (0.0, float(sig_mat.size)),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
